# News Article Summarizer

## 1. Project Overview

**Task:** Summarize news articles into concise, neutral daily summaries using three approaches: extractive (frequency-based), abstractive (BART transformer), and LLM-based (Ollama local model).

**Approach:** We compare three summarization paradigms — extractive (pick the best existing sentences), abstractive (generate new text with BART), and LLM prompting (instruction-following with a local model) — then evaluate all three with ROUGE.

**Stack:**
- `LangChain` + `ChatOllama` — LLM-based summarization via local Ollama
- `transformers` — BART abstractive summarization
- `rouge-score` — automated evaluation
- `Ollama` + `qwen3.5:9b` — local LLM (no cloud API keys)

> **No API keys required.** Everything runs locally via Ollama + Hugging Face.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Implement extractive summarization** using sentence scoring by word frequency |
| 2 | **Use BART** for abstractive summarization via Hugging Face `pipeline` |
| 3 | **Use a local LLM** (Ollama) for instruction-based summarization |
| 4 | **Evaluate** all methods with ROUGE-1, ROUGE-2, ROUGE-L |
| 5 | **Compare** extractive vs abstractive vs LLM tradeoffs |
| 6 | Design prompts for **neutral, factual** news summaries |
| 7 | Handle **length control** in summaries |
| 8 | Understand when each method excels or fails |

## 3. Problem Statement

News articles range from 300 to 3,000+ words. Readers want:
- A 2–4 sentence summary capturing the key facts
- Neutral tone — no editorializing
- Preservation of who, what, when, where, why

This is the foundation for news digests, alert systems, and media monitoring.

## 4. Why This Project Matters

- **High demand:** Google News, Apple News, and every news aggregator uses auto-summarization
- **Three paradigms in one project:** extractive, abstractive, and LLM-based — understanding all three is essential
- **ROUGE evaluation** is the standard NLP summarization metric — learning it here transfers everywhere
- **Neutrality prompting** teaches tone control, a key skill for responsible AI

## 5. Pipeline Architecture

```
News Article (raw text)
      │
      ├──▶ [Extractive] ── sentence scoring → top-k selection → ordered output
      │
      ├──▶ [BART Abstractive] ── encode → decode → generated summary
      │
      └──▶ [LLM Prompting] ── system prompt (neutral tone) → article → summary
      │
      ▼
  [Evaluate] ── ROUGE-1 / ROUGE-2 / ROUGE-L against reference
      │
      ▼
  [Compare] ── speed, quality, neutrality, coverage
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core transformers torch rouge-score

print("Dependencies: langchain, langchain-ollama, transformers, rouge-score")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")
print("Transformer model: facebook/bart-large-cnn (auto-downloaded)")

## 7. Imports & Configuration

In [ ]:
import os, json, re, time, warnings
import numpy as np
from textwrap import dedent

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0)


def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def ask(system: str, user: str) -> str:
    resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return clean(resp.content)


test = ask("Reply with one word.", "Say ready.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Sample News Articles

## 8. Create Sample Articles with Reference Summaries

We use realistic news articles across different topics (tech, health, economy, climate) with human-written reference summaries for ROUGE evaluation.

In [ ]:
ARTICLES = {
    "tech_layoffs": {
        "title": "Major Tech Company Announces 12,000 Layoffs Amid AI Pivot",
        "category": "Technology",
        "article": dedent("""Silicon Valley giant TechCorp announced on Wednesday that it will lay off approximately 12,000 employees, roughly 6% of its global workforce, as the company accelerates its shift toward artificial intelligence products. CEO Maria Chen said in an internal memo obtained by reporters that the cuts are necessary to \"refocus our resources on the AI opportunity that will define the next decade of computing.\"

The layoffs will primarily affect the company's cloud infrastructure division and its legacy advertising technology unit. Employees in affected roles will receive 16 weeks of severance pay, six months of health insurance continuation, and job placement assistance.

TechCorp's stock rose 4.2% in after-hours trading following the announcement, as investors viewed the restructuring as a positive signal for the company's AI strategy. The company reported spending $3.8 billion on AI research in the last fiscal year and plans to increase that to $5.5 billion in the coming year.

Industry analysts noted that TechCorp's move follows a broader trend among technology companies reallocating resources toward AI. \"This is a structural shift in how tech companies allocate capital,\" said Jennifer Walsh, a technology analyst at Goldman Sachs.

Labor advocates criticized the layoffs. The Tech Workers Union called on the company to invest in retraining programs. TechCorp responded by announcing a $50 million retraining fund for affected employees interested in AI and machine learning roles."""),
        "reference_summary": "TechCorp will lay off 12,000 employees (6% of its workforce) to redirect resources toward AI development. The cuts mainly affect cloud infrastructure and advertising technology divisions. Affected employees receive 16 weeks severance and six months health insurance. The company plans to increase AI research spending from $3.8 billion to $5.5 billion. TechCorp stock rose 4.2% on the news.",
    },
    "health_study": {
        "title": "Landmark Study Links Ultra-Processed Foods to Cognitive Decline",
        "category": "Health",
        "article": dedent("""A major longitudinal study published in The Lancet Neurology has found a significant association between high consumption of ultra-processed foods and accelerated cognitive decline in adults over 50. The study, conducted by researchers at the University of S\u00e3o Paulo and Harvard Medical School, followed 10,775 participants across eight countries over a period of 12 years.

Participants who derived more than 20% of their daily caloric intake from ultra-processed foods showed a 28% faster rate of cognitive decline compared to those who consumed minimal amounts.

Dr. Claudia Suemoto, the study's lead author, cautioned that the findings show correlation rather than definitive causation. \"We controlled for income, education, physical activity, smoking, and pre-existing conditions, but there may be unmeasured confounders,\" she said.

The study measured cognitive function using standardized tests of memory, executive function, and processing speed. The sharpest declines were observed in memory and executive function.

Dr. Walter Willett of Harvard called it \"the strongest evidence yet that diet quality in middle age may influence cognitive trajectory.\" He recommended that public health guidelines emphasize whole foods."""),
        "reference_summary": "A 12-year study of 10,775 adults published in The Lancet Neurology found that people getting over 20% of calories from ultra-processed foods experienced 28% faster cognitive decline. The study covered eight countries and controlled for lifestyle factors, though researchers note correlation rather than causation. Memory and executive function showed the sharpest declines.",
    },
    "economy_rates": {
        "title": "Federal Reserve Holds Interest Rates Steady, Signals Possible Cut in September",
        "category": "Economy",
        "article": dedent("""The Federal Reserve held its benchmark interest rate steady at 5.25-5.50% on Wednesday, as widely expected, but signaled that a rate cut could come as early as September if inflation continues to moderate. Fed Chair Jerome Powell said the committee \"is gaining greater confidence that inflation is moving sustainably toward 2%\" but wants to see \"a few more months of good data\" before acting.

The decision was unanimous among the 12 voting members. Economic activity \"has continued to expand at a solid pace\" while the labor market is \"gradually rebalancing.\" Inflation, as measured by PCE, fell to 2.5%, down from a peak of 7.1% in June 2022.

Markets reacted positively. The S&P 500 rose 1.1% and the 10-year Treasury yield fell 8 basis points to 4.12%. Traders now price in a 78% probability of a September rate cut.

Consumer spending has remained resilient despite high borrowing costs, but credit card delinquencies have climbed to levels not seen since 2011."""),
        "reference_summary": "The Federal Reserve held interest rates at 5.25-5.50% but signaled a possible September cut if inflation continues declining. PCE inflation fell to 2.5%, down from a peak of 7.1%. Markets reacted positively, with the S&P 500 rising 1.1% and traders pricing a 78% chance of a September cut. Credit card delinquencies have risen to 2011 levels.",
    },
    "climate_report": {
        "title": "UN Report: Global Temperatures on Track to Exceed 1.5\u00b0C Threshold by 2027",
        "category": "Climate",
        "article": dedent("""A new report from the United Nations World Meteorological Organization warns that there is a 66% probability that global average temperatures will temporarily exceed the 1.5\u00b0C warming threshold above pre-industrial levels in at least one year between now and 2027.

The 1.5\u00b0C limit was established as a key target in the 2015 Paris Agreement. Exceeding it is expected to intensify extreme weather events. 2023 was already the warmest year on record at 1.45\u00b0C above pre-industrial levels.

\"We are on uncharted territory,\" said WMO Secretary-General Celeste Saulo. She emphasized that a single year above 1.5\u00b0C does not mean the Paris target is permanently breached, but serves as a stark warning.

The report identifies El Ni\u00f1o as a contributing factor combined with human-caused warming. Climate scientists called for an immediate acceleration of renewable energy deployment and fossil fuel phase-down."""),
        "reference_summary": "A UN WMO report says there is a 66% chance global temperatures will temporarily exceed the 1.5\u00b0C Paris Agreement threshold between now and 2027. 2023 was the warmest year on record at 1.45\u00b0C above pre-industrial levels. El Ni\u00f1o and human emissions are both contributing factors. Scientists called for faster renewable energy deployment.",
    },
}

print(f"Loaded {len(ARTICLES)} articles:")
for key, art in ARTICLES.items():
    print(f"  {key:20s} | {art['category']:10s} | {len(art['article']):,d} chars")

---

# Part B — Three Summarization Approaches

## 9. Approach 1: Extractive Summarization (Word Frequency)

Select the most important sentences based on word-frequency scoring. Fast, no model needed.

In [ ]:
def extractive_summarize(text: str, n_sentences: int = 3) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    if len(sentences) <= n_sentences:
        return text.strip()

    stopwords = {"the","a","an","is","are","was","were","be","been","being","have","has","had","do","does","did","will","would","could","should","may","might","to","of","in","for","on","with","at","by","from","as","and","but","or","not","so","it","its","this","that","he","she","they","we","you","said","who","which","than","also"}
    word_freq = {}
    for sent in sentences:
        for word in re.findall(r'\w+', sent.lower()):
            if word not in stopwords and len(word) > 2:
                word_freq[word] = word_freq.get(word, 0) + 1

    scores = []
    for sent in sentences:
        words = re.findall(r'\w+', sent.lower())
        score = sum(word_freq.get(w, 0) for w in words if w not in stopwords)
        scores.append(score / (len(words) + 1))

    top_indices = sorted(sorted(range(len(scores)), key=lambda i: -scores[i])[:n_sentences])
    return " ".join(sentences[i] for i in top_indices)


art = ARTICLES["tech_layoffs"]
ext_summary = extractive_summarize(art["article"], n_sentences=3)
print("EXTRACTIVE SUMMARY — Tech Layoffs")
print("=" * 50)
print(ext_summary)
print(f"\nLength: {len(ext_summary.split())} words")

## 10. Approach 2: BART Abstractive Summarization

BART (`facebook/bart-large-cnn`) generates **new text** rather than selecting existing sentences.

In [ ]:
import torch
from transformers import pipeline as hf_pipeline

device_id = 0 if torch.cuda.is_available() else -1
print(f"BART device: {'GPU' if device_id == 0 else 'CPU'}")

bart_summarizer = hf_pipeline("summarization", model="facebook/bart-large-cnn", device=device_id)


def bart_summarize(text: str, max_length: int = 130, min_length: int = 40) -> str:
    try:
        result = bart_summarizer(text[:1024], max_length=max_length, min_length=min_length, do_sample=False)
        return result[0]["summary_text"]
    except Exception as e:
        return f"Error: {e}"


bart_summary = bart_summarize(art["article"])
print("BART SUMMARY — Tech Layoffs")
print("=" * 50)
print(bart_summary)

## 11. Approach 3: LLM-Based Summarization (Ollama)

A local LLM with a prompt enforcing neutrality, conciseness, and factual coverage.

In [ ]:
NEWS_SYSTEM = """You are a professional news editor who writes concise, neutral summaries.
Rules:
- Write exactly 3-5 sentences
- Cover the key facts: who, what, when, where, why
- Stay neutral — no opinions, no editorializing
- Include the most important numbers and data points
- Do not add information not in the article
- Return ONLY the summary text, no headers or labels"""

NEWS_USER = """Summarize this news article:

Title: {title}
Category: {category}

Article:
{article}

Summary:"""


def llm_summarize(key: str) -> str:
    art = ARTICLES[key]
    return ask(NEWS_SYSTEM, NEWS_USER.format(title=art["title"], category=art["category"], article=art["article"]))


llm_summary = llm_summarize("tech_layoffs")
print("LLM SUMMARY — Tech Layoffs")
print("=" * 50)
print(llm_summary)

## 12. Run All Three Methods on All Articles

In [ ]:
results = {}
for key in ARTICLES:
    art = ARTICLES[key]
    print(f"\nProcessing: {art['title'][:50]}...")

    t0 = time.perf_counter()
    ext = extractive_summarize(art["article"], n_sentences=3)
    ext_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    bart = bart_summarize(art["article"])
    bart_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    llm_out = llm_summarize(key)
    llm_time = time.perf_counter() - t0

    results[key] = {"extractive": ext, "bart": bart, "llm": llm_out,
                    "reference": art["reference_summary"],
                    "times": {"extractive": round(ext_time, 2), "bart": round(bart_time, 2), "llm": round(llm_time, 2)}}
    print(f"  Ext: {ext_time:.2f}s | BART: {bart_time:.2f}s | LLM: {llm_time:.2f}s")

print("\nAll articles processed!")

---

# Part C — Evaluation

## 13. ROUGE Score Computation

ROUGE measures word overlap between generated and reference summaries:
- **ROUGE-1:** unigram overlap — **ROUGE-2:** bigram overlap — **ROUGE-L:** longest common subsequence

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge(prediction: str, reference: str) -> dict:
    scores = scorer.score(reference, prediction)
    return {k: round(scores[k].fmeasure, 4) for k in ["rouge1", "rouge2", "rougeL"]}

print(f"{'Article':<20} {'Method':<12} {'ROUGE-1':>9} {'ROUGE-2':>9} {'ROUGE-L':>9}")
print("=" * 65)

all_scores = {"extractive": [], "bart": [], "llm": []}
for key in ARTICLES:
    ref = results[key]["reference"]
    for method in ["extractive", "bart", "llm"]:
        rouge = compute_rouge(results[key][method], ref)
        all_scores[method].append(rouge)
        print(f"{key:<20} {method:<12} {rouge['rouge1']:>9.4f} {rouge['rouge2']:>9.4f} {rouge['rougeL']:>9.4f}")
    print()

print("AVERAGES")
print("-" * 50)
for method in ["extractive", "bart", "llm"]:
    avg = {k: np.mean([s[k] for s in all_scores[method]]) for k in ["rouge1", "rouge2", "rougeL"]}
    print(f"  {method:<12} R1={avg['rouge1']:.4f}  R2={avg['rouge2']:.4f}  RL={avg['rougeL']:.4f}")

## 14. Speed vs Quality Comparison

In [ ]:
print("SPEED COMPARISON")
print("=" * 60)
print(f"{'Article':<20} {'Extractive':>12} {'BART':>12} {'LLM':>12}")
print("-" * 60)
totals = {"extractive": 0, "bart": 0, "llm": 0}
for key in ARTICLES:
    t = results[key]["times"]
    print(f"{key:<20} {t['extractive']:>10.2f}s {t['bart']:>10.2f}s {t['llm']:>10.2f}s")
    for m in totals: totals[m] += t[m]
print("-" * 60)
print(f"{'TOTAL':<20} {totals['extractive']:>10.2f}s {totals['bart']:>10.2f}s {totals['llm']:>10.2f}s")
print("\n\u2022 Extractive: fastest, no model needed, copies verbatim")
print("\u2022 BART: generates fluent text, limited to 1024-token input")
print("\u2022 LLM: most flexible, handles instructions, slowest")

---

## 15. Prompt Variations — How Design Affects Quality

In [ ]:
variants = {
    "Full (default)": NEWS_SYSTEM,
    "Minimal": "Summarize the following news article in 3-5 sentences.",
    "Bullet points": "Summarize this news article as exactly 5 bullet points. Each bullet should be one factual statement. Return ONLY the bullets.",
    "Executive brief": "You write executive briefings for a busy CEO. Lead with the most important takeaway. Include decision-driving numbers. 3 sentences maximum.",
}

test_art = ARTICLES["economy_rates"]
user_prompt = f"Article:\n{test_art['article']}\n\nSummary:"

print("PROMPT VARIANT COMPARISON")
print("=" * 60)
for name, sys_prompt in variants.items():
    summary = ask(sys_prompt, user_prompt)
    rouge = compute_rouge(summary, test_art["reference_summary"])
    print(f"\n--- {name} ---")
    print(summary[:300])
    print(f"  ROUGE-L: {rouge['rougeL']:.4f} | Words: {len(summary.split())}")

## 16. Side-by-Side Output Comparison

In [ ]:
for key in ["health_study", "climate_report"]:
    art = ARTICLES[key]
    r = results[key]
    print(f"\n{'=' * 70}")
    print(f"ARTICLE: {art['title']}")
    print("=" * 70)
    print(f"\n\ud83d\udcf0 REFERENCE:\n  {r['reference'].strip()}")
    print(f"\n\u2702\ufe0f  EXTRACTIVE:\n  {r['extractive'][:300]}")
    print(f"\n\ud83e\udd16 BART:\n  {r['bart'][:300]}")
    print(f"\n\ud83d\udcac LLM:\n  {r['llm'][:300]}")

## 17. Error Analysis — Edge Cases

In [ ]:
print("EDGE CASE 1: Very short article")
print("-" * 50)
short = "Apple released iOS 18.5 today with bug fixes and performance improvements. The update is available for iPhone 12 and later."
print(f"  Extractive: {extractive_summarize(short, 3)}")
print(f"  LLM: {ask(NEWS_SYSTEM, f'Article:{short}')}")
print("  \u2192 LLM should not pad with invented details\n")

print("EDGE CASE 2: Number-heavy article")
print("-" * 50)
nums = "Revenue was $12.3B, up 18% YoY. Net income $2.1B vs $1.8B. Operating margin expanded from 14.2% to 17.1%. Added 3.2M subscribers to 45.7M total. ARPU grew 12% to $27.40. Stock rose 6.3% after-hours."
print(f"  LLM: {ask(NEWS_SYSTEM, f'Article:{nums}')}")
print("  \u2192 Should preserve the most important numbers\n")

print("EDGE CASE 3: Controversial topic")
print("-" * 50)
controversial = "The city council voted 7-4 to approve a housing development. Supporters say 500 affordable units and 200 jobs. Opponents say 30% more traffic, loss of historic park, and the developer donated $50K to three council members."
print(f"  LLM: {ask(NEWS_SYSTEM, f'Article:{controversial}')}")
print("  \u2192 Should present both sides neutrally")

## 18. Common Mistakes

| Mistake | Why It's Wrong | Fix |
|---------|---------------|-----|
| **ROUGE-only evaluation** | Misses fluency, factuality, neutrality | Add LLM judge or human eval |
| **No length control** | Summaries vary wildly | Specify sentence count in prompt |
| **Truncating articles blindly** | Loses late-article info | Use chunked/map-reduce |
| **Extractive for conciseness** | Copies full sentences verbatim | Use abstractive or LLM |
| **Ignoring neutrality** | Biased summaries erode trust | Add neutrality checks |
| **Not comparing methods** | Each has different strengths | Always benchmark |

## 19. Production Ideas

1. **RSS feed integration** — continuous summarization of incoming news
2. **Multi-article digest** — cluster related articles into topic digests
3. **Headline generation** — produce both summary and headline
4. **Entity tracking** — track people and companies across summaries
5. **Translation + summarization** — summarize foreign-language articles in English
6. **Fact verification** — cross-reference claims against other sources

## 20. Exercises

In [ ]:
# Exercise 1: Multi-article daily digest
DIGEST_SYS = """Combine these individual news summaries into one cohesive daily digest.
8-10 sentences. Smooth transitions between topics. Neutral tone."""

all_sums = "\n\n".join(f"[{ARTICLES[k]['category']}] {ARTICLES[k]['title']}:\n{results[k]['llm']}" for k in ARTICLES)
digest = ask(DIGEST_SYS, f"Summaries:\n{all_sums}\n\nDigest:")
print("DAILY NEWS DIGEST")
print("=" * 50)
print(digest)

In [ ]:
# Exercise 2: Headline generation
HEAD_SYS = "Generate a concise news headline (max 15 words). Factual, not clickbait. Return ONLY the headline."
print("HEADLINE GENERATION")
print("=" * 50)
for key in ARTICLES:
    art = ARTICLES[key]
    headline = ask(HEAD_SYS, f"Article:\n{art['article'][:500]}\n\nHeadline:")
    print(f"  Original:  {art['title']}")
    print(f"  Generated: {headline}\n")

### Mini Challenge

1. **TF-IDF extractive:** Replace word frequency with TF-IDF. Does ROUGE improve?
2. **Prompt ablation:** Remove neutrality rules. How does output change for the controversial article?
3. **Model comparison:** Try a different Ollama model and compare ROUGE scores.
4. **Real article:** Paste a real news article and run all three methods.

## 21. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)
print(f"Articles processed: {len(ARTICLES)}")
print(f"Methods: extractive, BART abstractive, LLM (Ollama)")
print(f"Evaluation: ROUGE-1, ROUGE-2, ROUGE-L")
print(f"Prompt variants tested: 4")
print(f"Edge cases tested: 3")
print(f"\nAverage ROUGE-L:")
for method in ["extractive", "bart", "llm"]:
    avg = np.mean([s["rougeL"] for s in all_scores[method]])
    print(f"  {method:12s}: {avg:.4f}")
print(f"\nLLM: {LLM_MODEL}")

## 22. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Three paradigms** — extractive (fast, copies), abstractive (fluent, limited context), LLM (flexible, slowest) |
| 2 | **ROUGE is necessary but insufficient** — captures overlap, misses factuality and neutrality |
| 3 | **Prompt design matters** — neutrality rules and sentence-count targets measurably improve LLM output |
| 4 | **BART excels at conciseness** — trained specifically for summarization |
| 5 | **Extractive is a strong baseline** — zero model, instant results |
| 6 | **LLMs handle format flexibility** — bullets, briefs, digests from the same model |
| 7 | **Neutrality must be checked** — no method guarantees neutral output by default |
| 8 | **Length control is hard** — LLMs may over- or under-produce despite instructions |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*